# Stage 09 — Cross-file trajectory continuity audit

**Research question:** Does processing every GeoLife .plt independently cause valid stays to be missed at file boundaries?

This is an experimental A/B evaluation, **not a production migration**. Both arms use the frozen CP1 settings: 10 m same-second consolidation, 300 s gap, 1200 km/h speed guard, 200 m anchor radius, and 1200 s dwell. The only change is whether admissible neighboring files from the **same user** are cleaned and detected together.

1. Reproduce frozen 5,821 stays / 136 users and HOME 27 / OFFICE 16 from the existing Volume cache.
2. Inventory file endpoints and audit **every adjacent file boundary**. Never bridge overlaps, equal timestamps, impossible speeds, invalid coordinates, or excessive gaps.
3. Rerun original CP1 preprocessing/detection only on safe adjacent-file components. Limit memory and explicitly report skipped large components.
4. Compare stay/user coverage and frozen semantic policy on the experimental inventory. No threshold change, no automatic production promotion.

**Key limitation:** A connection may be continuous but still not contain a stay. A connection with endpoint separation over 400 m cannot keep both endpoints within the same 200 m anchor radius (triangle inequality); this notebook uses that necessary geometric condition to focus reprocessing on boundaries capable of hosting a cross-file stay. This is a targeted audit, not a blanket attempt to reconstruct every continuous user trajectory.

User-level and coordinate-bearing outputs remain **private on Modal Volume**. Only aggregate summaries should be shared.

In [ ]:
from pathlib import Path
from datetime import timezone
from hashlib import sha256
import gc, importlib.util, json, os, subprocess, sys

import numpy as np
import pandas as pd
from IPython.display import display

REPO_URL = "https://github.com/tanh1c/geolife.git"
BRANCH = os.environ.get("GEOLIFE_REPO_BRANCH", "experiment/09-cross-file-continuity-audit")
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/root/geolife"))
V = Path("/mnt/geolife-data")
C = V / "cache" / "cp2_v2" / "09_cross_file_continuity_audit"
C.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    subprocess.run(cmd, cwd=cwd, check=True)

if not (REPO_DIR / ".git").exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH])
    run(["git", "-C", str(REPO_DIR), "checkout", BRANCH])
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH])

run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], cwd=REPO_DIR)
for p in (str(REPO_DIR), str(REPO_DIR / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

from geolife.staypoints import clean_trajectory, detect_staypoints
from geolife.geo.distance import haversine_m
from geolife.model import HomeOfficeConfig, build_semantic_locations, infer_home_office

SHA = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip()
print("Git commit:", SHA)
print("Modal Volume:", V)
print("Stage 09 cache:", C)

## 1. Frozen inputs and experimental contract

The frozen baseline is loaded, **not recomputed**. The full GeoLife release is expected to contain 182 users and 18,670 .plt files.

Change MAX_GROUP_RAW_POINTS only as a resource safeguard; it is **not** a semantic threshold. Changing any CP1 parameter invalidates this experiment as a pure file-boundary comparison.

In [ ]:
CP1 = dict(same_second_radius_m=10.0, max_gap_s=300.0, hard_speed_guard_kmh=1200.0)
STAY = dict(distance_threshold_m=200.0, min_dwell_s=1200.0)
CACHE_SCHEMA = "09a_v1"
MAX_GROUP_RAW_POINTS = 500_000  # skip larger groups; retain their unchanged baseline stays

def find_raw_root():
    env = os.environ.get("GEOLIFE_RAW_ROOT") or os.environ.get("GEOLIFE_DATA_ROOT")
    candidates = ([Path(env)] if env else []) + [
        V / "extracted" / "Geolife Trajectories 1.3" / "Data",
        V / "Data",
        V / "Geolife Trajectories 1.3" / "Data",
    ]
    for p in candidates:
        if p.is_dir() and (p / "000" / "Trajectory").is_dir():
            return p.resolve()
    for p in sorted(V.glob("**/Geolife Trajectories 1.3/Data")):
        if (p / "000" / "Trajectory").is_dir():
            return p.resolve()
    raise FileNotFoundError("GeoLife raw Data/ root not found. Set GEOLIFE_RAW_ROOT.")

def find_stay_cache():
    candidates = [
        V / "cache" / "03a_user_behavior_deep_dive" / "stays_baseline_v1.pkl",
        V / "cache" / "cp2_home_office" / "stays_baseline_v1.pkl",
        V / "artifacts" / "03a" / "stays_baseline_v1.pkl",
    ]
    for p in candidates:
        if p.exists():
            return p
    hits = sorted(V.glob("**/stays_baseline_v1.pkl"))
    if hits:
        return hits[0]
    raise FileNotFoundError("Missing frozen stays_baseline_v1.pkl on the Modal Volume.")

RAW_ROOT = find_raw_root()
FILES = sorted(RAW_ROOT.glob("*/Trajectory/*.plt"))
USER_IDS = sorted(p.name for p in RAW_ROOT.iterdir() if p.is_dir() and p.name.isdigit())
assert len(USER_IDS) == 182 and len(FILES) == 18670, (
    f"Unexpected release: users={len(USER_IDS)} files={len(FILES)}"
)

STAY_CACHE = find_stay_cache()
baseline = pd.read_pickle(STAY_CACHE).copy()
baseline["user_id"] = baseline["user_id"].astype(str).str.zfill(3)
baseline["file_key"] = list(zip(baseline["user_id"], baseline["source_file"].map(lambda x: Path(str(x)).name)))
assert len(baseline) == 5821 and baseline["user_id"].nunique() == 136
cfg = HomeOfficeConfig()
production = infer_home_office(baseline.drop(columns="file_key"), config=cfg)
prod_counts = production["label"].value_counts().to_dict()
assert prod_counts.get("HOME",0)==27 and prod_counts.get("OFFICE",0)==16
print("Frozen parity: PASS | stays=5821 | stay users=136 | HOME=27 | OFFICE=16")
print("Raw root:", RAW_ROOT, "| frozen cache:", STAY_CACHE)

## 2. Raw file endpoint inventory — resumable

A quick line scan records first/last raw observation and file point count without materializing a 24.9M-point DataFrame. The private manifest is cached every 100 .plt files and resumes after interruption. The reader deliberately does not drop invalid endpoint coordinates to manufacture continuity.

Every file is identified by **(user_id, filename)**; this is how the frozen stay cache's source-file lineage is reconciled with the extracted release.

In [ ]:
from collections import deque

PLT_NAMES = ["latitude", "longitude", "unused", "altitude_ft", "serial_date", "date", "time"]
MANIFEST_CACHE = C / f"raw_file_endpoints_{CACHE_SCHEMA}.pkl"
META_CONTEXT = C / f"raw_file_endpoints_{CACHE_SCHEMA}_context.json"

def parse_gps_row(line):
    if isinstance(line, bytes):
        line = line.decode("utf-8", errors="replace")
    fields = line.strip().split(",")
    if len(fields) < 7:
        return (pd.NaT, np.nan, np.nan)
    ts = pd.to_datetime(fields[5] + " " + fields[6], utc=True, errors="coerce")
    try:
        lat, lon = float(fields[0]), float(fields[1])
    except ValueError:
        lat, lon = np.nan, np.nan
    return (ts, lat, lon)

def file_endpoint_meta(path):
    """Read only the first GPS row and a small tail; do NOT scan every GPS row."""
    stat = path.stat()
    with path.open("rb") as f:
        for _ in range(6):
            f.readline()
        first = f.readline()
        # All GeoLife PLT records are short; 8 KiB includes the terminal record.
        f.seek(max(0, stat.st_size - 8192))
        tail = f.read()
    last_rows = tail.splitlines()
    last = last_rows[-1] if last_rows else b""
    a, alat, alon = parse_gps_row(first)
    b, blat, blon = parse_gps_row(last)
    return dict(
        user_id=path.parent.parent.name, filename=path.name,
        filepath=str(path), n_raw_points=pd.NA,  # deferred: count ONLY candidate component files
        file_bytes=stat.st_size, file_mtime_ns=stat.st_mtime_ns,
        first_ts=a, first_lat=alat, first_lon=alon,
        last_ts=b, last_lat=blat, last_lon=blon,
    )

expected_context = dict(schema=CACHE_SCHEMA, raw_root=str(RAW_ROOT), files=len(FILES))
if MANIFEST_CACHE.exists() and META_CONTEXT.exists() and json.loads(META_CONTEXT.read_text()) == expected_context:
    manifest = pd.read_pickle(MANIFEST_CACHE)
    rows = manifest.to_dict("records")
    seen = set(zip(manifest["user_id"], manifest["filename"]))
    print(f"RESUMED: {len(rows):,}/{len(FILES):,} existing file endpoints")
else:
    rows, seen = [], set()

for ix, path in enumerate(FILES, 1):
    key = (path.parent.parent.name, path.name)
    if key in seen:
        continue
    rows.append(file_endpoint_meta(path))
    seen.add(key)
    if len(seen) % 250 == 0:
        manifest = pd.DataFrame.from_records(rows)
        temp = MANIFEST_CACHE.with_suffix(".partial.pkl")
        manifest.to_pickle(temp)
        os.replace(temp, MANIFEST_CACHE)
        META_CONTEXT.write_text(json.dumps(expected_context))
        print(f"Endpoint inventory: {len(seen):,}/{len(FILES):,} files", flush=True)

manifest = pd.DataFrame.from_records(rows)
assert len(manifest) == len(FILES)
assert not manifest.duplicated(["user_id", "filename"]).any()
manifest["first_ts"] = pd.to_datetime(manifest["first_ts"], utc=True)
manifest["last_ts"] = pd.to_datetime(manifest["last_ts"], utc=True)
manifest.to_pickle(MANIFEST_CACHE)
META_CONTEXT.write_text(json.dumps(expected_context))
assert set(baseline["file_key"]).issubset(
    set(zip(manifest["user_id"], manifest["filename"]))
)
print("Endpoint manifest: PASS",len(manifest),"files")
print("GPS point counts are intentionally deferred to candidate components (Stage 4).")
display(manifest[["file_bytes"]].describe().round(1))

## 3. Audit adjacent .plt boundaries

Within each user, files are ordered by actual first timestamp (not filename). A boundary is eligible for **targeted cross-file stay testing** only when all conditions hold:

- positive time separation, at most 300 s;
- valid first and last endpoint coordinates;
- implied speed at most 1200 km/h;
- endpoint separation at most 400 m (necessary, **not sufficient**, for one 200 m-anchor stay to span that boundary).

Zero-gap timestamps, overlaps, invalid endpoints, and distant/no-coverage boundaries are preserved as separate diagnostic categories. No labels are inferred in this step.

In [ ]:
def valid_coord(lat,lon):
    return (np.isfinite(lat) and np.isfinite(lon) and -90 <= lat <= 90 and -180 <= lon <= 180)

def boundary_record(a,b):
    out=dict(user_id=str(a.user_id), left_file=a.filename, right_file=b.filename,
             left_last_ts=a.last_ts, right_first_ts=b.first_ts,
             gap_s=np.nan, endpoint_distance_m=np.nan, speed_kmh=np.nan)
    if pd.isna(a.last_ts) or pd.isna(b.first_ts):
        out["status"]="invalid_endpoint_time"
        return out
    if (not valid_coord(a.last_lat,a.last_lon) or not valid_coord(b.first_lat,b.first_lon)):
        out["status"]="invalid_endpoint_coordinate"
        return out
    gap=float((b.first_ts-a.last_ts).total_seconds())
    out["gap_s"]=gap
    if gap < 0:
        out["status"]="overlapping_files"
        return out
    if gap == 0:
        out["status"]="same_timestamp_boundary"
        return out
    if gap > CP1["max_gap_s"]:
        out["status"]="gap_over_300s"
        return out
    d=float(haversine_m(a.last_lat,a.last_lon,b.first_lat,b.first_lon))
    out["endpoint_distance_m"]=d
    speed=d/gap*3.6
    out["speed_kmh"]=speed
    if speed > CP1["hard_speed_guard_kmh"]:
        out["status"]="speed_guard"
    elif d > 2*STAY["distance_threshold_m"]:
        out["status"]="too_far_for_shared_anchor"
    else:
        out["status"]="stitch_candidate"
    return out

ordered = manifest.sort_values(["user_id","first_ts","filename"],kind="stable",na_position="last").reset_index(drop=True)
boundary_rows=[]
for uid,g in ordered.groupby("user_id",sort=True):
    for ix in range(len(g)-1):
        boundary_rows.append(boundary_record(g.iloc[ix],g.iloc[ix+1]))
boundaries=pd.DataFrame(boundary_rows)
assert len(boundaries)==len(FILES)-len(USER_IDS)
assert not boundaries.duplicated(["user_id","left_file","right_file"]).any()
boundaries.to_pickle(C/"file_boundary_audit_private.pkl")
boundary_summary=boundaries["status"].value_counts().rename_axis("status").reset_index(name="boundaries")
boundary_summary.to_csv(C/"boundary_status_summary.csv",index=False)
display(boundary_summary)
print("Potentially stitchable boundaries:",int(boundaries["status"].eq("stitch_candidate").sum()))

## 4. Build safe connected file components

Only consecutive files linked by stitch_candidate are grouped. Disconnected files retain their original frozen CP1 stays. We rerun the **same functions** clean_trajectory and detect_staypoints on each group, and never join different users.

Long connected components are skipped if their raw GPS point count exceeds MAX_GROUP_RAW_POINTS. A skip is reported separately: it cannot be interpreted as no impact.

In [ ]:
links = set(zip(
    boundaries.loc[boundaries.status.eq("stitch_candidate"), "user_id"],
    boundaries.loc[boundaries.status.eq("stitch_candidate"), "left_file"],
    boundaries.loc[boundaries.status.eq("stitch_candidate"), "right_file"],
))
components = []
for uid, g in ordered.groupby("user_id", sort=True):
    rows = list(g.itertuples(index=False))
    current = [rows[0]]
    for r in rows[1:]:
        if (uid, current[-1].filename, r.filename) in links:
            current.append(r)
        else:
            if len(current) > 1:
                components.append((uid, list(current)))
            current = [r]
    if len(current) > 1:
        components.append((uid, list(current)))

# Count exact raw points ONLY for files in eligible components. Streaming byte
# chunks is substantially faster than iterating 25 million rows in Python.
POINT_COUNT_CACHE = C / f"candidate_file_point_counts_{CACHE_SCHEMA}.pkl"
counts = pd.read_pickle(POINT_COUNT_CACHE) if POINT_COUNT_CACHE.exists() else {}
def count_raw_gps_points(path):
    n, last_byte = 0, b""
    with Path(path).open("rb") as f:
        for _ in range(6):
            f.readline()
        while True:
            chunk = f.read(4 * 1024 * 1024)
            if not chunk:
                break
            n += chunk.count(b"\n")
            last_byte = chunk[-1:]
    return n + int(bool(last_byte) and last_byte != b"\n")

pending = 0
def get_n_points(row):
    global pending
    if pd.notna(row.n_raw_points):
        return int(row.n_raw_points)   # existing checkpoint from original notebook
    key = (row.user_id, row.filename, int(row.file_bytes), int(row.file_mtime_ns))
    if key not in counts:
        counts[key] = count_raw_gps_points(row.filepath)
        pending += 1
        if pending % 25 == 0:
            pd.to_pickle(counts, POINT_COUNT_CACHE)
            print(f"Counted {pending} new candidate files", flush=True)
    return int(counts[key])

plans = []
for idx, (uid, rows) in enumerate(components, 1):
    plans.append(dict(
        component_id=f"G{idx:05d}", user_id=uid, n_files=len(rows),
        n_raw_points=sum(get_n_points(x) for x in rows),
        filenames=[x.filename for x in rows],
        filepaths=[x.filepath for x in rows],
        fingerprint=sha256("|".join(
            f"{x.filename}:{x.file_bytes}:{x.file_mtime_ns}" for x in rows
        ).encode()).hexdigest()[:12],
    ))
pd.to_pickle(counts, POINT_COUNT_CACHE)

plan = pd.DataFrame(plans)
if plan.empty:
    print("No eligible components. Frozen baseline unchanged under this rule.")
else:
    plan["too_large"] = plan.n_raw_points > MAX_GROUP_RAW_POINTS
    print("Components:",len(plan),"| skipped by resource cap:",int(plan.too_large.sum()))
    display(plan[["n_files", "n_raw_points"]].describe().round(1))
plan.to_pickle(C / "stitch_component_plan_private.pkl")

## 5. Synthetic safety checks

Run these before processing the real candidate components. The valid example is a 25-minute stay split across two files, which cannot satisfy the 20-minute dwell threshold when processed independently.

The regression tests also cover a >300 s gap and a same-timestamp boundary. These are **synthetic checks**, not measured GeoLife results.

In [ ]:
def synthetic_frame(times, lat=39., lon=116.):
    return pd.DataFrame({
        "timestamp": pd.to_datetime(times, utc=True),
        "latitude": lat, "longitude": lon,
    })

# Intervals inside files must be <=300 s. Previous example used 6-minute
# intervals and was therefore INVALID for frozen CP1 continuity.
a = synthetic_frame([
    "2026-01-01 08:00:00", "2026-01-01 08:04:00",
    "2026-01-01 08:08:00", "2026-01-01 08:12:00",
])
b = synthetic_frame([
    "2026-01-01 08:13:00", "2026-01-01 08:17:00",
    "2026-01-01 08:21:00", "2026-01-01 08:25:00",
])
assert detect_staypoints(clean_trajectory(a, **CP1), **STAY).empty
assert detect_staypoints(clean_trajectory(b, **CP1), **STAY).empty
joined = detect_staypoints(
    clean_trajectory(pd.concat([a, b], ignore_index=True), **CP1), **STAY
)
assert len(joined) == 1 and int(joined.iloc[0].duration_s) == 1500

def pseudo(frame, name, uid="000"):
    return pd.Series(dict(
        user_id=uid, filename=name,
        first_ts=frame.timestamp.iloc[0],
        first_lat=frame.latitude.iloc[0], first_lon=frame.longitude.iloc[0],
        last_ts=frame.timestamp.iloc[-1],
        last_lat=frame.latitude.iloc[-1], last_lon=frame.longitude.iloc[-1],
    ))
assert boundary_record(pseudo(a, "a.plt"), pseudo(b, "b.plt"))["status"] == "stitch_candidate"
late = synthetic_frame(["2026-01-01 08:19:00", "2026-01-01 08:23:00"])
assert boundary_record(pseudo(a, "a.plt"), pseudo(late, "late.plt"))["status"] == "gap_over_300s"
same = synthetic_frame(["2026-01-01 08:12:00", "2026-01-01 08:16:00"])
assert boundary_record(pseudo(a, "a.plt"), pseudo(same, "same.plt"))["status"] == "same_timestamp_boundary"
print("Synthetic self-check: PASS")

## 6. Rerun eligible components (per-component cache)

Each group is reread from the same raw GeoLife release, its points concatenated only for that group, and frozen CP1 functions rerun unchanged. Completed groups are cached individually. Raw GPS arrays are discarded immediately; only derived stays and diagnostics are cached.

For every group we count stays that **span an actual file boundary**. This is stronger evidence for a file-splitting effect than a simple increase in total stay count.

Do not change the cache schema or CP1 configuration without invalidating previous component caches.

In [ ]:
GROUP_CACHE=C/"groups"
GROUP_CACHE.mkdir(parents=True,exist_ok=True)
STAY_COLUMNS=["user_id","source_file","sequence_id","arrival_time_utc","departure_time_utc","duration_s","latitude","longitude","n_points"]

def read_raw_plt(path):
    raw=pd.read_csv(path,skiprows=6,header=None,names=PLT_NAMES,usecols=[0,1,5,6])
    raw["timestamp"]=pd.to_datetime(raw["date"].astype(str)+" "+raw["time"].astype(str),
                                   format="%Y-%m-%d %H:%M:%S",utc=True,errors="coerce")
    if raw["timestamp"].isna().any():
        raise ValueError(f"Unparsable timestamp in {Path(path).name}")
    return raw[["timestamp","latitude","longitude"]]

def run_component(item):
    paths=list(item.filepaths)
    raw=pd.concat([read_raw_plt(p) for p in paths],ignore_index=True)
    cleaned=clean_trajectory(raw,**CP1)
    ds=detect_staypoints(cleaned,**STAY)
    if ds.empty:
        detected=pd.DataFrame(columns=STAY_COLUMNS)
    else:
        detected=ds.rename(columns={"arrival_time":"arrival_time_utc",
                                     "departure_time":"departure_time_utc"})
        detected["user_id"]=item.user_id
        detected["source_file"]=f"STITCHED:{item.component_id}"
        detected=detected[STAY_COLUMNS].copy()
    edges=boundaries.loc[
        boundaries.user_id.eq(item.user_id)
        & boundaries.left_file.isin(item.filenames)
        & boundaries.right_file.isin(item.filenames)
        & boundaries.status.eq("stitch_candidate")
    ]
    n_spanning=0
    if not detected.empty:
        for stay in detected.itertuples(index=False):
            if any(
                stay.arrival_time_utc<=edge.left_last_ts
                and stay.departure_time_utc>=edge.right_first_ts
                for edge in edges.itertuples(index=False)
            ):
                n_spanning+=1
    del raw,cleaned,ds
    gc.collect()
    return {"stays":detected,"spanning_stays":n_spanning,"component_id":item.component_id,
            "user_id":item.user_id,"fingerprint":item.fingerprint}

processed=[]
skipped=[]
cached_hits=0
for ix,item in enumerate(plan.itertuples(index=False),1):
    if item.n_raw_points>MAX_GROUP_RAW_POINTS:
        skipped.append(dict(component_id=item.component_id,user_id=item.user_id,
                            n_files=item.n_files,n_raw_points=item.n_raw_points,
                            reason="exceeds_group_point_cap"))
        continue
    path=GROUP_CACHE/f"{CACHE_SCHEMA}_{item.component_id}_{item.fingerprint}_cp1fixed.pkl"
    if path.exists():
        payload=pd.read_pickle(path)
        cached_hits+=1
    else:
        payload=run_component(item)
        temp=path.with_suffix(".partial.pkl")
        pd.to_pickle(payload,temp)
        os.replace(temp,path)
    processed.append(payload)
    if ix%10==0 or ix==len(plan):
        print(f"Components {ix}/{len(plan)} | cached={cached_hits} | skipped={len(skipped)}")
print("Recomputed:",len(processed),"components | skipped:",len(skipped))
pd.DataFrame(skipped).to_csv(C/"resource_skips_private.csv",index=False)

## 7. Coverage comparison — full release, with honest skip accounting

The experimental inventory replaces baseline stays only for **successfully rerun groups**. All unprocessed/oversized groups keep their frozen baseline rows. Report both net count change and how many stitched stays span a file boundary. An increase in stays alone cannot establish better detection accuracy.

In [ ]:
affected=set()
for item in plan.itertuples(index=False):
    if item.component_id in {p["component_id"] for p in processed}:
        affected.update((item.user_id,name) for name in item.filenames)

unchanged=baseline.loc[~baseline.file_key.isin(affected)].drop(columns="file_key")
new_parts=[p["stays"] for p in processed if not p["stays"].empty]
new_stays=pd.concat(new_parts,ignore_index=True) if new_parts else pd.DataFrame(columns=STAY_COLUMNS)
effective=pd.concat([unchanged[STAY_COLUMNS],new_stays[STAY_COLUMNS]],ignore_index=True)
assert not effective["user_id"].isna().any()
baseline_user_counts=baseline.groupby("user_id").size().reindex(USER_IDS,fill_value=0)
effective_user_counts=effective.groupby("user_id").size().reindex(USER_IDS,fill_value=0)
user_delta=(effective_user_counts-baseline_user_counts).rename("delta_stays")
spanning=sum(p["spanning_stays"] for p in processed)

component_results=[]
for item in plan.itertuples(index=False):
    matching=[p for p in processed if p["component_id"]==item.component_id]
    keys={(item.user_id,name) for name in item.filenames}
    old_count=int(baseline.file_key.isin(keys).sum())
    if matching:
        payload=matching[0]
        component_results.append(dict(component_id=item.component_id,user_id=item.user_id,
            status="evaluated",old_stays=old_count,new_stays=len(payload["stays"]),
            delta_stays=len(payload["stays"])-old_count,
            spanning_stays=payload["spanning_stays"],n_files=item.n_files))
    else:
        component_results.append(dict(component_id=item.component_id,user_id=item.user_id,
            status="skipped_size",old_stays=old_count,new_stays=old_count,
            delta_stays=0,spanning_stays=0,n_files=item.n_files))
result_groups=pd.DataFrame(component_results)
result_groups.to_pickle(C/"component_comparison_private.pkl")
user_delta.to_frame().to_pickle(C/"user_delta_private.pkl")
effective.to_pickle(C/"experimental_stays_private.pkl")

impact=pd.DataFrame([{
    "baseline_stays":len(baseline),
    "experimental_stays":len(effective),
    "delta_stays":len(effective)-len(baseline),
    "baseline_stay_users":int((baseline_user_counts>0).sum()),
    "experimental_stay_users":int((effective_user_counts>0).sum()),
    "new_stay_users":int(((baseline_user_counts==0)&(effective_user_counts>0)).sum()),
    "lost_stay_users":int(((baseline_user_counts>0)&(effective_user_counts==0)).sum()),
    "users_with_more_stays":int((user_delta>0).sum()),
    "users_with_fewer_stays":int((user_delta<0).sum()),
    "cross_file_spanning_stays":int(spanning),
    "stitch_candidates":int(boundaries.status.eq("stitch_candidate").sum()),
    "components_evaluated":len(processed),
    "components_skipped":len(skipped)
}])
impact.to_csv(C/"stay_impact_summary.csv",index=False)
display(impact.T.rename(columns={0:"value"}))
if not result_groups.empty:
    display(result_groups.groupby("status")[["old_stays","new_stays","delta_stays","spanning_stays"]].sum())

## 8. Downstream effect: recurring locations and HOME/OFFICE

Apply the **unchanged** production HomeOfficeConfig to the experimental stay inventory. Spatial correspondence (within 200 m) is used for user-label comparisons; integer location IDs cannot be compared between two different clusterings.

This diagnostic is not semantic ground truth. Unchanged outputs are a useful robustness check; additional emissions are **experimental**.

In [ ]:
def summarize_semantics(stays):
    ss,ll=build_semantic_locations(stays,config=cfg)
    outputs=infer_home_office(stays,config=cfg)
    # infer_home_office() already returns latitude/longitude.
    # Merging locations again would introduce latitude_x/latitude_y suffixes.
    assert {"user_id","location_id","label","latitude","longitude"}.issubset(outputs.columns)
    labeled=outputs.copy()
    return {
        "semantic_stays":len(ss),
        "locations":len(ll),
        "recurring_locations":int(ll.stay_count.ge(2).sum()),
        "recurring_users":int(ll.loc[ll.stay_count.ge(2),"user_id"].nunique()),
        "HOME":int(outputs.label.eq("HOME").sum()),
        "OFFICE":int(outputs.label.eq("OFFICE").sum()),
    },labeled

ref_summary,ref_labels=summarize_semantics(baseline.drop(columns="file_key"))
new_summary,new_labels=summarize_semantics(effective)
assert ref_summary["HOME"]==27 and ref_summary["OFFICE"]==16
comparison=pd.DataFrame({"frozen":ref_summary,"experimental":new_summary})
comparison["delta"]=comparison["experimental"]-comparison["frozen"]
comparison.to_csv(C/"semantic_impact_summary.csv")
display(comparison)

rows=[]
for label in ("HOME","OFFICE"):
    aa=ref_labels.loc[ref_labels.label.eq(label)].set_index("user_id")
    bb=new_labels.loc[new_labels.label.eq(label)].set_index("user_id")
    common=aa.index.intersection(bb.index)
    matched=0
    for uid in common:
        a,b=aa.loc[uid],bb.loc[uid]
        if float(haversine_m(a.latitude,a.longitude,b.latitude,b.longitude))<=200:
            matched+=1
    rows.append(dict(label=label,baseline=len(aa),experimental=len(bb),
                     both_selected=len(common),both_within_200m=matched,
                     baseline_only=len(aa.index.difference(bb.index)),
                     experimental_only=len(bb.index.difference(aa.index))))
semantic_match=pd.DataFrame(rows)
semantic_match.to_csv(C/"semantic_correspondence_summary.csv",index=False)
display(semantic_match)

## 9. Interpretation and next decision

**Read outputs in order:** boundary categories -> eligible components / skipped components -> actual spanning stays -> net stay/user change -> recurring locations -> HOME/OFFICE.

- If few actual stays span boundaries, file splitting is not a major measured driver of frozen low coverage under the 300 s / 200 m / 20 min contract; it does not rule out the effect of strict thresholds or missed ambiguous overlaps.
- If many spanning stays are recovered, consider a separate validated ingestion/stitching contract before changing semantic gates.
- If the experiment changes candidate locations or HOME/OFFICE, manually audit representative cases in the private Volume and use stability tests before any production change.
- Skipped large groups are **unmeasured**, never zero effect. Same-timestamp and overlapping boundaries are **deliberately unresolved** in this stage.
- Counts/agreements are not accuracy metrics; there is no direct HOME/OFFICE ground truth.

**Volume outputs:** private manifest, boundary table, component plan/results, experimental stays, user deltas; shareable aggregate CSVs boundary_status_summary.csv, stay_impact_summary.csv, semantic_impact_summary.csv and semantic_correspondence_summary.csv.

**Production policy stays frozen:** HOME 27 / OFFICE 16. No production source files or existing caches are modified.